### Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from kaggledatahandler import KaggleDataHandler
from audiopreprocessing import SoundDS
import numpy as np
import os
import datetime
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix


### Creating dataset (Training & Test)

In [2]:
KDHandler = KaggleDataHandler()
datasets_filepath_organized, y_datasets_filepath_organized = KDHandler.create_set()
print("Created the following filepaths with correspondong label\n:", datasets_filepath_organized.keys())


Created the following filepaths with correspondong label
: dict_keys(['fold1', 'fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10'])


In [3]:

preprocessed_datasets = {}
audiopreprocessor = SoundDS()

for fold in datasets_filepath_organized:
    new_prepro_fold = []
    for i, file in enumerate(datasets_filepath_organized[fold]):
        class_ID = y_datasets_filepath_organized[fold][i]
        filepath = file
        spectrgram, class_id = audiopreprocessor.__getitem__(filepath, class_ID)
        preprocessed_wav = (spectrgram, class_id)
        new_prepro_fold.append(preprocessed_wav)
    print("Finished", fold)
    preprocessed_datasets[fold] = new_prepro_fold

Finished fold1
Finished fold2
Finished fold3
Finished fold4
Finished fold5
Finished fold6
Finished fold7
Finished fold8
Finished fold9
Finished fold10


### CNN Model

In [4]:
class AudioClassifier(nn.Module):

    def __init__(self):
        super().__init__()
        conv_layers = []

        # First Convolution Block
        self.conv1 = nn.Conv2d(2, 8, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
        self.relu1 = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(8)
        init.kaiming_normal_(self.conv1.weight, a=0.1)
        self.conv1.bias.data.zero_()
        conv_layers += [self.conv1, self.relu1, self.bn1]

        # Second Convolution Block
        self.conv2 = nn.Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu2 = nn.ReLU()
        self.bn2 = nn.BatchNorm2d(16)
        init.kaiming_normal_(self.conv2.weight, a=0.1)
        self.conv2.bias.data.zero_()
        conv_layers += [self.conv2, self.relu2, self.bn2]

        # Third Convolution Block
        self.conv3 = nn.Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu3 = nn.ReLU()
        self.bn3 = nn.BatchNorm2d(32)
        init.kaiming_normal_(self.conv3.weight, a=0.1)
        self.conv3.bias.data.zero_()
        conv_layers += [self.conv3, self.relu3, self.bn3]

        # Fourth Convolution Block
        self.conv4 = nn.Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu4 = nn.ReLU()
        self.bn4 = nn.BatchNorm2d(64)
        init.kaiming_normal_(self.conv4.weight, a=0.1)
        self.conv4.bias.data.zero_()
        conv_layers += [self.conv4, self.relu4, self.bn4]

        # Convolution blocks wrapped in Sequential
        self.conv = nn.Sequential(*conv_layers)

        # Adaptive Pooling
        self.ap = nn.AdaptiveAvgPool2d(output_size=1)

        # Dropout (p = 0.5 is common)
        self.dropout = nn.Dropout(p=0.5)

        # Linear Classifier
        self.lin = nn.Linear(64, 10)

    def forward(self, x):
        x = self.conv(x)
        x = self.ap(x)
        x = x.view(x.shape[0], -1)

        x = self.dropout(x)

        x = self.lin(x)
        return x


### Testing loop

In [5]:

def test(model, val_dl):
  correct_prediction = 0
  total_prediction = 0

  all_labels = []
  all_preds = []

  # Disable gradient updates
  with torch.no_grad():
    for data in val_dl:
      # Get the input features and target labels, and put them on the GPU
      inputs, labels = data[0].to(device), data[1].to(device)

      # Normalize the inputs
      inputs_m, inputs_s = inputs.mean(), inputs.std()
      inputs = (inputs - inputs_m) / inputs_s

      # Get predictions
      outputs = model(inputs)

      # Get the predicted class with the highest score
      _, prediction = torch.max(outputs,1)
      
      all_labels.append(labels.cpu())
      all_preds.append(prediction.cpu())
      # Count of predictions that matched the target label
      correct_prediction += (prediction == labels).sum().item()
      total_prediction += prediction.shape[0]
    
  acc = correct_prediction/total_prediction
  all_labels = all_labels
  all_preds = torch.cat(all_preds)
  print(f'Accuracy: {acc:.2f}, Total items: {total_prediction}\n')
  return acc, all_labels, all_preds


### Training loop

In [6]:
def training(model, train_dl, num_epochs, device):
  # Loss Function, Optimizer and Scheduler
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
  scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001,
                                                steps_per_epoch=int(len(train_dl)),
                                                epochs=num_epochs,
                                                anneal_strategy='linear')
  
  losses = []
  accuracies = []

  # Repeat for each epoch
  for epoch in range(num_epochs):
    running_loss = 0.0
    correct_prediction = 0
    total_prediction = 0

    # Repeat for each batch in the training set
    for i, data in enumerate(train_dl):
        # Get the input features and target labels, and put them on the GPU
        inputs, labels = data[0].to(device), data[1].to(device)

        # Normalize the inputs
        inputs_m, inputs_s = inputs.mean(), inputs.std()
        inputs = (inputs - inputs_m) / inputs_s

        # Zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Keep stats for Loss and Accuracy
        running_loss += loss.item()

        # Get the predicted class with the highest score
        _, prediction = torch.max(outputs,1)
        # Count of predictions that matched the target label
        correct_prediction += (prediction == labels).sum().item()
        total_prediction += prediction.shape[0]

        #if i % 10 == 0:    # print every 10 mini-batches
        #    print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / 10))
    
    # Print stats at the end of the epoch
    num_batches = len(train_dl)
    avg_loss = running_loss / num_batches
    acc = correct_prediction/total_prediction
    losses.append(avg_loss)
    accuracies.append(acc)

    print(f'Epoch: {epoch}, Loss: {avg_loss:.2f}, Accuracy: {acc:.2f}')

  print('Finished Training\n')
  return losses, accuracies

### Ploting function

In [7]:
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np   # needed for confusion matrices & mean handling

def plot_and_save_cv_curves(
    train_accuracies,
    train_losses,
    test_accuracies,
    test_accuracy_labels=None,
    confusion_matrices=None,
    class_names=None,
    base_dir="results"
):
    # 1. Create main result directory
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)

    # 2. Create timestamped folder
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    result_path = os.path.join(base_dir, timestamp)
    os.makedirs(result_path)

    # 3. Create training and test subfolders
    train_path = os.path.join(result_path, "training")
    test_path = os.path.join(result_path, "test")
    os.makedirs(train_path)
    os.makedirs(test_path)

    print(f"Saving all plots to: {result_path}")

    # ---------------------------
    # 4. Training curves - ALL folds in ONE figure
    # ---------------------------
    plt.figure(figsize=(10, 4))
    ax_loss = plt.subplot(1, 2, 1)
    ax_acc = plt.subplot(1, 2, 2)

    for fold_idx in sorted(train_accuracies.keys()):
        accs = train_accuracies[fold_idx]
        losses = train_losses[fold_idx]
        epochs = list(range(1, len(accs) + 1))

        # Loss curves for all folds
        ax_loss.plot(epochs, losses, marker='o', label=f"Fold {fold_idx + 1}")

        # Accuracy curves for all folds
        ax_acc.plot(epochs, accs, marker='o', label=f"Fold {fold_idx + 1}")

    ax_loss.set_title("Training Loss per Epoch (All Folds)")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.grid(True)
    ax_loss.legend()

    ax_acc.set_title("Training Accuracy per Epoch (All Folds)")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.grid(True)
    ax_acc.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(train_path, "training_curves_all_folds.png"))
    plt.close()

    # ----------------------------------------------
    # 5. Test accuracy histogram
    # ----------------------------------------------

    fold_labels = []
    fold_values = []

    for fold_idx in sorted(test_accuracies.keys()):
        # Get label
        if test_accuracy_labels:
            label_list = test_accuracy_labels.get(fold_idx)
            label = label_list[0] if isinstance(label_list, list) else label_list
        else:
            label = f"Fold {fold_idx + 1}"

        # Always store labels as strings
        fold_labels.append(str(label))

        # Get accuracy data for this fold
        value = test_accuracies[fold_idx]

        # If it's a list/tuple/array, reduce to a single scalar
        if isinstance(value, (list, tuple, np.ndarray)):
            if len(value) == 1:
                scalar = value[0]
            else:
                # e.g. multiple test accuracies per epoch -> take mean
                scalar = np.mean(value)
        else:
            scalar = value

        fold_values.append(float(scalar))

    # Histogram / bar chart of test accuracies per fold
    plt.figure(figsize=(7, 5))
    x_pos = np.arange(len(fold_values))
    plt.bar(x_pos, fold_values)
    plt.title("Test Accuracy per Fold")
    plt.xlabel("Fold")
    plt.ylabel("Accuracy")
    plt.xticks(x_pos, fold_labels, rotation=45, ha="right")
    plt.grid(axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(test_path, "test_accuracy_histogram.png"))
    plt.close()

    # ----------------------------------------------
    # 6. Boxplot of test accuracies across folds
    # ----------------------------------------------
    plt.figure(figsize=(6, 5))
    plt.boxplot(fold_values, vert=True)
    plt.title("Distribution of Test Accuracies Across Folds")
    plt.ylabel("Accuracy")
    plt.xticks([1], ["Test Accuracies"])  # single box label
    plt.grid(axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(test_path, "test_accuracy_boxplot.png"))
    plt.close()

    # ----------------------------------------------
    # 7. Confusion matrices per fold (OPTIONAL)
    # ----------------------------------------------
    if confusion_matrices is not None:
        cm_path = os.path.join(test_path, "confusion_matrices")
        os.makedirs(cm_path, exist_ok=True)

        for fold_idx in sorted(confusion_matrices.keys()):
            cm = np.array(confusion_matrices[fold_idx])

            plt.figure(figsize=(6, 5))
            im = plt.imshow(cm, interpolation='nearest')
            plt.title(f"Confusion Matrix - Fold {fold_idx + 1}")
            plt.colorbar(im)
            plt.xlabel("Predicted")
            plt.ylabel("True")

            # Tick labels if class_names are provided
            if class_names is not None:
                plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
                plt.yticks(np.arange(len(class_names)), class_names)
            else:
                plt.xticks(np.arange(cm.shape[1]))
                plt.yticks(np.arange(cm.shape[0]))

            # Values inside each cell
            thresh = cm.max() / 2.0 if cm.size > 0 else 0
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    plt.text(
                        j, i, str(cm[i, j]),
                        ha="center", va="center",
                        color="white" if cm[i, j] > thresh else "black"
                    )

            plt.tight_layout()
            plt.savefig(os.path.join(cm_path, f"confusion_matrix_fold_{fold_idx+1}.png"))
            plt.close()

    print("Plots saved successfully.")
    return result_path


### Training & Test

In [ ]:
number_of_test_folds = 1
sets = KDHandler.create_splits(number_of_test_folds)
models = []
num_epochs = 5

training_losses_all = {}
training_accuracies_all = {}
test_accuracies_all = {}
test_accuracies_all_fold = {}
confusion_matrices_all = {}
   


for i, combination_set in enumerate(sets):

    # Create the model and put it on the GPU if available
    new_model = AudioClassifier()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    new_model = new_model.to(device)
    # Check that it is on Cuda
    next(new_model.parameters()).device

    print(f"##### Set {i} ####\n")

    print(f" - Starting training for set #{i}\n")
    train_set =[]
    for fold in combination_set[1]:
        if len(train_set) == 0:
            train_set = preprocessed_datasets[fold]
        else:
            train_set = train_set + preprocessed_datasets[fold]
 
    print(f"Training from the folds {combination_set[1]}\n")
    train_dl = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=False)
    losses, accuracies = training(new_model, train_dl, num_epochs, device)
    training_losses_all[i] = losses
    training_accuracies_all[i] = accuracies
    
    print(f"Starting testing for set #{i}\n")
    new_test_acc = []
    new_test_acc_fold = []   
    for fold in combination_set[0]:
        print(f"Testing on fold #{fold}")
        test_dl = torch.utils.data.DataLoader(preprocessed_datasets[fold], batch_size=16, shuffle=False)  
        acc, fold, all_preds = test(new_model, test_dl)
        cm = confusion_matrix(torch.cat(fold), all_preds)
        new_test_acc.append(acc)
        new_test_acc_fold.append(fold)
        confusion_matrices_all[i] = cm
    test_accuracies_all[i] = new_test_acc
    test_accuracies_all_fold[i] = new_test_acc_fold

    models.append(new_model)
    

##### Set 0 ####

 - Starting training for set #0

Training from the folds ['fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10']

Epoch: 0, Loss: 2.02, Accuracy: 0.27
Epoch: 1, Loss: 1.65, Accuracy: 0.42
Epoch: 2, Loss: 1.49, Accuracy: 0.48
Epoch: 3, Loss: 1.39, Accuracy: 0.52
Epoch: 4, Loss: 1.35, Accuracy: 0.54
Finished Training

Starting testing for set #0

Testing on fold #fold1
Accuracy: 0.49, Total items: 873

##### Set 1 ####

 - Starting training for set #1

Training from the folds ['fold1', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10']

Epoch: 0, Loss: 2.06, Accuracy: 0.26
Epoch: 1, Loss: 1.66, Accuracy: 0.42
Epoch: 2, Loss: 1.46, Accuracy: 0.50
Epoch: 3, Loss: 1.34, Accuracy: 0.54
Epoch: 4, Loss: 1.31, Accuracy: 0.55
Finished Training

Starting testing for set #1

Testing on fold #fold2
Accuracy: 0.47, Total items: 888

##### Set 2 ####

 - Starting training for set #2

Training from the folds ['fold1', 'fold2', 'fold4',

### Results

In [ ]:
result_dir = plot_and_save_cv_curves(
    train_accuracies=training_accuracies_all,
    train_losses=training_losses_all,
    test_accuracies=test_accuracies_all,
    test_accuracy_labels=test_accuracies_all_fold,
    confusion_matrices=confusion_matrices_all,
    class_names=None,                           
    base_dir="results"
)


Saving all plots to: results/2025-11-26_17-48-43


/tmp/ipykernel_1461388/1721495454.py:102: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Plots saved successfully.
